# Full LSTM training — Google Colab (T4)

Trains the LSTM at the project's target config (embedding 256, hidden 512, 2 layers, dropout 0.2, 8 epochs) on ≈300k filter-passing games from one Lichess monthly dump. Expected wall-clock on a free T4: ≈1–2 hours.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Push the project to GitHub and edit `GITHUB_URL` below. (Alt path via Drive upload is documented in cell 4.)
3. Have a Weights & Biases API key handy if you want experiment tracking (free tier is fine).

**What this notebook produces and persists to Drive:**
- `lstm_full.pt` — best-by-val-loss checkpoint
- `vocab.json` — vocab built from the full train split
- `eval_test.txt` — the eval harness's top-1/3/5 + perplexity + phase breakdown on the test split


## 1. GPU + environment sanity

In [ ]:
!nvidia-smi

## 2. Mount Drive (for persisting outputs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DRIVE_DIR = '/content/drive/MyDrive/chess-move-prediction'
!mkdir -p "$PROJECT_DRIVE_DIR"
print('Outputs will be persisted to:', PROJECT_DRIVE_DIR)

## 3. Get the project code

**Default: clone from GitHub.** Edit `GITHUB_URL` to your repo.

**Alt: from Drive.** Zip the project locally, upload to Drive, then run instead of the cell below:
```python
!cp "$PROJECT_DRIVE_DIR/chess-move-prediction.zip" /content/
!cd /content && unzip -q chess-move-prediction.zip
```

In [ ]:
GITHUB_URL = 'https://github.com/kornel9/chess-move-prediction'  # <-- edit me

!cd /content && rm -rf chess-move-prediction && git clone $GITHUB_URL chess-move-prediction
%cd /content/chess-move-prediction
!git rev-parse --short HEAD

## 4. Install dependencies

In [ ]:
# Colab already ships torch + numpy + pandas. Install only what's missing.
!pip install -q chess==1.11.2 zstandard==0.25.0 wandb==0.26.1
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 5. Download a Lichess monthly and slice to 300k filter-passing games

Default: **2017-01** (≈1.9 GB compressed, comfortably contains ≥300k games passing our filters: Rapid/Classical, both Elo ≥ 1500, normal termination, 40–240 plies). Pick a different month at <https://database.lichess.org/> if you want.

We stream-decompress and slice in one pass via `src/data/make_smoke_slice.py` (the slicer reuses the existing PGN filters), so we never need the full decompressed file on disk.

In [ ]:
MONTHLY = '2017-01'
N_GAMES = 300000

import os
os.makedirs('data/raw', exist_ok=True)
DUMP = f'data/raw/lichess_{MONTHLY}.pgn.zst'
URL = f'https://database.lichess.org/standard/lichess_db_standard_rated_{MONTHLY}.pgn.zst'

if not os.path.exists(DUMP):
    !curl -L -o $DUMP $URL
!ls -lh $DUMP

In [ ]:
!python -m src.data.make_smoke_slice \
    --src $DUMP \
    --dst data/raw/full.pgn \
    --n $N_GAMES
!ls -lh data/raw/full.pgn

## 6. (Optional) Weights & Biases login

Skip this cell if you don't want W&B logging — just drop `--wandb` from the training command in cell 7.

In [ ]:
import wandb
wandb.login()  # paste API key from https://wandb.ai/authorize

## 7. Train the LSTM (full config)

Defaults match the project target: embedding 256, hidden 512, 2 layers, dropout 0.2, 8 epochs, AdamW + ReduceLROnPlateau, grad clip 1.0. Best-by-val-loss checkpoint saved to `checkpoints/full/lstm.pt`.

In [ ]:
!mkdir -p checkpoints/full
!python -m src.training.train_lstm \
    --pgn data/raw/full.pgn \
    --vocab data/vocab.json \
    --out checkpoints/full/lstm.pt \
    --epochs 8 \
    --batch-size 64 \
    --embedding-dim 256 \
    --hidden-dim 512 \
    --num-layers 2 \
    --dropout 0.2 \
    --num-workers 2 \
    --device cuda \
    --wandb

## 8. Evaluate on the test split

In [ ]:
!python -m src.training.evaluate \
    --model-type lstm \
    --model checkpoints/full/lstm.pt \
    --pgn data/raw/full.pgn \
    --vocab data/vocab.json \
    --split test \
    --device cuda \
    | tee eval_test.txt

## 9. (Optional) Train and eval the n-gram on the same split

Useful for an apples-to-apples comparison — the n-gram runs in pure Python, no GPU needed, and gives the rubric's baseline number to compare against.

In [ ]:
!python -m src.training.train_ngram \
    --pgn data/raw/full.pgn \
    --vocab data/vocab.json \
    --out checkpoints/full/ngram.pkl.gz

!python -m src.training.evaluate \
    --model-type ngram \
    --model checkpoints/full/ngram.pkl.gz \
    --pgn data/raw/full.pgn \
    --vocab data/vocab.json \
    --split test \
    | tee eval_test_ngram.txt

## 10. Persist outputs to Drive

In [ ]:
!cp checkpoints/full/lstm.pt        "$PROJECT_DRIVE_DIR/lstm_full.pt"
!cp data/vocab.json                 "$PROJECT_DRIVE_DIR/vocab.json"
!cp eval_test.txt                   "$PROJECT_DRIVE_DIR/eval_test_lstm.txt"
# n-gram artefacts only exist if cell 9 ran
!test -f checkpoints/full/ngram.pkl.gz && cp checkpoints/full/ngram.pkl.gz "$PROJECT_DRIVE_DIR/ngram_full.pkl.gz" || true
!test -f eval_test_ngram.txt && cp eval_test_ngram.txt "$PROJECT_DRIVE_DIR/eval_test_ngram.txt" || true
!ls -lh "$PROJECT_DRIVE_DIR"

## 11. What to bring back to your local repo

Once the run finishes, copy the following into your local `WORKFLOW.md` so the engineering journal stays current:

1. The contents of `eval_test_lstm.txt` (and `eval_test_ngram.txt` if cell 9 ran).
2. The W&B run URL (printed at the start of training if `--wandb` was used).
3. Total wall-clock time (Colab prints per-epoch times; sum them).
4. Any surprises — errors, val loss plateaus, OOMs.

Also download `lstm_full.pt` and `vocab.json` from Drive into the local repo's `checkpoints/` and `data/` directories if you want to use the model locally (e.g. for the Streamlit demo).